In [1]:
import requests
import pandas as pd
import sqlite3
from ratelimit import limits, sleep_and_retry

WARFRAME_MARKET_API = "https://api.warframe.market/v2"
WARFRAME_MARKET_API_V1 = "https://api.warframe.market/v1"


def get_items():
    return requests.get(f"{WARFRAME_MARKET_API}/items").json()['data']

@sleep_and_retry
@limits(calls=10, period=1)
def get_stats(item):
    return requests.get(f"{WARFRAME_MARKET_API_V1}/items/{item}/statistics").json()[
        "payload"
    ]

def generate_db():
    with sqlite3.connect("./assets/data.db") as conn:
        c = conn.cursor()

        c.execute("DROP TABLE IF EXISTS arcanes;")
        c.execute("DROP TABLE IF EXISTS statistics;")

        c.execute("PRAGMA foreign_keys = 1")
        c.execute("""
            CREATE TABLE IF NOT EXISTS arcanes (
                name VARCHAR(255) PRIMARY KEY,
                max_rank int,
                rarity VARCHAR(255),
                type VARCHAR(255)
            );
        """)

        c.execute("""
            CREATE TABLE IF NOT EXISTS statistics (
                item_name VARCHAR(255) NOT NULL,
                datetime DATETIME,
                volume INT,
                min_price INT,
                max_price INT,
                open_price INT,
                closed_price INT,
                mod_rank INT,
                FOREIGN KEY (item_name) REFERENCES arcanes(name),
                PRIMARY KEY (datetime, mod_rank, item_name)
            );
        """)

        conn.commit()

def convert_to_slug(name: str):
    sep = "-" if "-" in name else "_"
    return name.replace(" ", sep).lower()

def generate_arcanes_data():
    with sqlite3.connect("./assets/data.db") as conn:
        res = pd.read_json("./assets/arcanes.json", orient="index")
        res[["Name", "MaxRank", "Rarity", "Type"]].rename(
            columns={
                "Name": "name",
                "MaxRank": "max_rank",
                "Rarity": "rarity",
                "Type": "type",
            }
        ).assign(name=lambda df: df["name"].map(convert_to_slug)).to_sql(
            "arcanes", conn, if_exists="append", index=False
        )

def generate_statistics_data():
    with sqlite3.connect("./assets/data.db") as conn:
        res = pd.read_sql_query("SELECT name FROM arcanes;", conn)
        c = conn.cursor()
        c.execute("PRAGMA foreign_keys = ON")

        for i, arcane in res.iterrows():
            data = get_stats(arcane["name"])
            statistics_closed = pd.DataFrame(data["statistics_closed"]["90days"])
            df = statistics_closed[
                [
                    "datetime",
                    "volume",
                    "min_price",
                    "max_price",
                    "open_price",
                    "closed_price",
                    "mod_rank",
                ]
            ].assign(item_name=arcane["name"])
            query = """
            INSERT INTO statistics (datetime, volume, min_price, max_price, open_price, closed_price, mod_rank, item_name)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            ON CONFLICT(datetime, mod_rank, item_name) DO UPDATE SET
                volume=excluded.volume,
                min_price=excluded.min_price,
                max_price=excluded.max_price,
                open_price=excluded.open_price,
                closed_price=excluded.closed_price
            """
            c.executemany(
                query,
                df[
                    [
                        "datetime",
                        "volume",
                        "min_price",
                        "max_price",
                        "open_price",
                        "closed_price",
                        "mod_rank",
                        "item_name",
                    ]
                ].itertuples(index=False, name=None),
            )
